Code để train AI, dùng trên collab drive

Buoc 1

In [ ]:
from google.colab import drive as o_dia
import os

o_dia.mount('/content/drive', force_remount=True)

duong_dan_du_lieu = '/content/drive/MyDrive/data'

if not os.path.exists(duong_dan_du_lieu):
    os.makedirs(duong_dan_du_lieu)

Mounted at /content/drive


Buoc 2

In [ ]:
from google.colab import drive
import os



# 2. Lệnh giải nén đúng đường dẫn thư mục PBL 5 của bạn
# Lưu ý: Dùng dấu ngoặc kép "" vì thư mục PBL 5 có dấu cách
!unzip -q "/content/drive/MyDrive/PBL 5/ocr_dataset.zip" -d "/content/dataset"

print("Đã giải nén thành công vào thư mục /content/dataset!")

Đã giải nén thành công vào thư mục /content/dataset!


Buoc 3

In [ ]:
import os
import random

def tao_nhan_tong_hop(cac_thu_muc, file_ra):
    with open(file_ra, 'w', encoding='utf-8') as f_tong:
        for tm in cac_thu_muc:
            if not os.path.exists(tm): continue
            cac_tep = os.listdir(tm)
            anh = [t for t in cac_tep if t.endswith('.jpg')]
            for ten_anh in anh:
                ten_goc = os.path.splitext(ten_anh)[0]
                tep_txt = os.path.join(tm, ten_goc + '.txt')
                if os.path.exists(tep_txt):
                    with open(tep_txt, 'r', encoding='utf-8') as f:
                        chu = f.read().strip()
                    f_tong.write(f"{os.path.join(tm, ten_anh)}\t{chu}\n")

def chia_hoc_thi(nguon, hoc, thi, ti_le=0.9):
    with open(nguon, 'r', encoding='utf-8') as f:
        dong = f.readlines()
    random.shuffle(dong)
    cat = int(len(dong) * ti_le)
    with open(hoc, 'w', encoding='utf-8') as f: f.writelines(dong[:cat])
    with open(thi, 'w', encoding='utf-8') as f: f.writelines(dong[cat:])

# Quét 2 thư mục tiếng Việt trong dataset của bạn
danh_sach_tm = ['/content/dataset/vi_00', '/content/dataset/vi_01']
tao_nhan_tong_hop(danh_sach_tm, '/content/dataset/tong.txt')
chia_hoc_thi('/content/dataset/tong.txt', '/content/dataset/hoc.txt', '/content/dataset/thi.txt')
print("Đã chuẩn bị xong file nhãn!")

Đã chuẩn bị xong file nhãn!


Buoc 4

In [ ]:
!pip install vietocr einops "numpy<2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Buoc 5

In [ ]:
import os
import shutil
import unicodedata
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

# 1. Dọn dẹp thư mục tạm để tránh lỗi dữ liệu cũ
for thu_muc_rac in ['train_data', 'valid_data']:
    if os.path.exists(thu_muc_rac):
        shutil.rmtree(thu_muc_rac)

# 2. Định nghĩa bộ từ vựng tiếng Việt
chu_cai_tieng_viet = "aàảãáạăằẳẵắặâầẩẫấậbcdđeèẻẽéẹêềểễếệghiìỉĩíịklmnoòỏõóọôồổỗốộơờởỡớợpqrstuùủũúụưừửữứựvxyỳỷỹýỵAÀẢÃÁẠĂẰẲẴẮẶÂẦẨẪẤẬBCDĐEÈẺẼÉẸÊỀỂỄẾỆGHIÌỈĨÍỊKLMNOÒỎÕÓỌÔỒỔỐỘƠỜỞỠỚỢPQRSTUÙỦŨÚỤƯỪỬỮỨỰVXYỲỶỸÝỴ0123456789!\"#$%&'()*+,-./:;<=>?@[\\]^_{|}~ "

# 3. Hàm chuẩn hóa văn bản
def chuan_hoa_van_ban(duong_dan_tep):
    dong_du_lieu = []
    with open(duong_dan_tep, 'r', encoding='utf-8') as f:
        for dong in f:
            phan_doan = dong.split('\t')
            if len(phan_doan) < 2: continue
            anh, chu = phan_doan[0], phan_doan[1].strip()
            chu = unicodedata.normalize('NFC', chu)
            chu = ''.join([k for k in chu if k in chu_cai_tieng_viet])
            if chu:
                dong_du_lieu.append(f"{anh}\t{chu}\n")
    with open(duong_dan_tep, 'w', encoding='utf-8') as f:
        f.writelines(dong_du_lieu)

# Đường dẫn file nhãn
tep_huan_luyen = '/content/dataset/hoc.txt'
tep_kiem_tra = '/content/dataset/thi.txt'
chuan_hoa_van_ban(tep_huan_luyen)
chuan_hoa_van_ban(tep_kiem_tra)

# 4. Cấu hình đường dẫn lưu Model Seq2Seq
duong_dan_luu_tru = '/content/drive/MyDrive/data/seq2seq_ocr_nhom11.pth'

# 5. Thiết lập bộ khung VGG_SEQ2SEQ (Thay vì transformer)
bo_khung = Cfg.load_config_from_name('vgg_seq2seq')
bo_khung['dataset']['vocab'] = chu_cai_tieng_viet
bo_khung['dataset']['train_annotation'] = tep_huan_luyen
bo_khung['dataset']['valid_annotation'] = tep_kiem_tra

# Tham số huấn luyện
bo_khung['trainer']['iters'] = 15000         # Seq2Seq hội tụ nhanh, 15k là ổn
bo_khung['trainer']['batch_size'] = 32
bo_khung['trainer']['print_every'] = 100
bo_khung['trainer']['save_every'] = 500      # Lưu mỗi 500 lượt để theo dõi

bo_khung['trainer']['checkpoint'] = duong_dan_luu_tru
bo_khung['trainer']['export'] = duong_dan_luu_tru
bo_khung['trainer']['metrics'] = 1000        # Cứ 1000 lượt sẽ đánh giá và lưu bản tốt nhất

# 6. Khởi tạo may_hoc với pretrained=True để dùng model gốc VietOCR
may_hoc = Trainer(bo_khung, pretrained=True)

# Kiểm tra nếu đã có file trên Drive thì học tiếp, không thì học từ bản gốc
if os.path.exists(duong_dan_luu_tru):
    may_hoc.config['trainer']['resume'] = duong_dan_luu_tru

# 7. Cơ chế thông báo khi lưu file xịn nhất (như đã làm với bản Trans)
ham_luu_goc = may_hoc.save_weights

def thong_bao_khi_luu(duong_dan):
    ham_luu_goc(duong_dan)
    print(f"\n---> DA LUU PHIEN BAN XIN NHAT VAO: {duong_dan} <---\n")

may_hoc.save_weights = thong_bao_khi_luu

# 8. Bắt đầu huấn luyện
may_hoc.train()

Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:08<00:00, 64.0MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
18533it [00:13, 1336.56it/s]
Create train_data: 100%|███████████████████████████████████| 179996/179996 [03:09<00:00, 952.18it/s]

Created dataset with 179995 samples



train_data build cluster: 100%|██████████████████████████| 179995/179995 [00:02<00:00, 71390.78it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 3 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Create valid_data: 100%|████████████████████████████████████| 20000/20000 [00:18<00:00, 1088.89it/s]

Created dataset with 19999 samples



valid_data build cluster: 100%|███████████████████████████| 19999/19999 [00:00<00:00, 146227.15it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 3 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


iter: 000100 - train loss: 0.801 - lr: 1.38e-05 - load time: 0.65 - gpu time: 49.09
iter: 000200 - train loss: 0.664 - lr: 1.91e-05 - load time: 0.05 - gpu time: 48.49
iter: 000300 - train loss: 0.630 - lr: 2.77e-05 - load time: 0.06 - gpu time: 46.89
iter: 000400 - train loss: 0.620 - lr: 3.95e-05 - load time: 0.05 - gpu time: 42.43
iter: 000500 - train loss: 0.611 - lr: 5.42e-05 - load time: 0.05 - gpu time: 49.55
iter: 000600 - train loss: 0.597 - lr: 7.14e-05 - load time: 0.06 - gpu time: 44.40
iter: 000700 - train loss: 0.602 - lr: 9.07e-05 - load time: 0.06 - gpu time: 51.65
iter: 000800 - train loss: 0.596 - lr: 1.12e-04 - load time: 0.06 - gpu time: 43.11
iter: 000900 - train loss: 0.591 - lr: 1.34e-04 - load time: 0.06 - gpu time: 42.83
iter: 001000 - train loss: 0.588 - lr: 1.56e-04 - load time: 0.06 - gpu time: 45.21
iter: 001100 - train loss: 0.586 - lr: 1.79e-04 - load time: 0.08 - gpu time: 45.71
iter: 001200 - train loss: 0.598 - lr: 2.01e-04 - load time: 0.07 - gpu time

In [1]:


import yaml
from vietocr.tool.config import Cfg

chu_cai_tieng_viet = "aàảãáạăằẳẵắặâầẩẫấậbcdđeèẻẽéẹêềểễếệghiìỉĩíịklmnoòỏõóọôồổỗốộơờởỡớợpqrstuùủũúụưừửữứựvxyỳỷỹýỵAÀẢÃÁẠĂẰẲẴẮẶÂẦẨẪẤẬBCDĐEÈẺẼÉẸÊỀỂỄẾỆGHIÌỈĨÍỊKLMNOÒỎÕÓỌÔỒỔỐỘƠỜỞỠỚỢPQRSTUÙỦŨÚỤƯỪỬỮỨỰVXYỲỶỸÝỴ0123456789!\"#$%&'()*+,-./:;<=>?@[\\]^_{|}~ "
bo_khung = Cfg.load_config_from_name('vgg_transformer')
bo_khung['dataset']['vocab'] = chu_cai_tieng_viet

duong_dan_cau_hinh = '/content/drive/MyDrive/data/cau_hinh_ocr.yml'
with open(duong_dan_cau_hinh, 'w', encoding='utf-8') as tap_tin:
    yaml.dump(dict(bo_khung), tap_tin, allow_unicode=True)

noi_dung_ma_nguon = """
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

def thiet_lap_bo_nhan_dang(duong_dan_cau_hinh, duong_dan_trong_so):
    cau_hinh_he_thong = Cfg.load_config_from_file(duong_dan_cau_hinh)
    cau_hinh_he_thong['weights'] = duong_dan_trong_so
    cau_hinh_he_thong['device'] = 'cpu'
    return Predictor(cau_hinh_he_thong)

def thuc_hien_nhan_dien(anh_dau_vao, may_ocr):
    ket_qua_chu_viet = may_ocr.predict(anh_dau_vao)
    return ket_qua_chu_viet
"""

with open('/content/drive/MyDrive/data/bien_dich_ocr.py', 'w', encoding='utf-8') as tap_tin_ma:
    tap_tin_ma.write(noi_dung_ma_nguon.strip())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')